# Visual Pipeline

1. Get Image
2. DETECTOR: Detect objects
3. SoM Painter: Get SoM Labeled Images
4. SGG: Get predicates and attributes
5. ReID: Get embeddings of image crops
6. Update Memory

In [ ]:

import matplotlib.pyplot as plt


def show(img_array, title=""):
    plt.figure(figsize=(10, 8))
    plt.imshow(img_array)
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
import logging

logger = logging.getLogger(__name__)


## Detection

In [ ]:
from pydantic import BaseModel
from pydantic import Field


class DetectionObject(BaseModel):
    class_id: int = Field(..., description="Class ID")
    label: str = Field(..., description="Object label")
    confidence: float = Field(..., description="Detection confidence")
    bbox: list[float] = Field(..., description="[x1, y1, x2, y2]")
    object_id: int | None = Field(None, description="Persistent tracking ID")

In [ ]:
from abc import ABC
from abc import abstractmethod

from PIL import Image
from rfdetr.detr import RFDETR
from rfdetr.util.coco_classes import COCO_CLASSES
from ultralytics.engine.model import Model


# TODO: maybe set to float16, no need for float32
class BaseDetector(ABC):
    @abstractmethod
    def predict(self, image: Image.Image) -> list[DetectionObject]:
        pass


class UltralyticsDetector(BaseDetector):
    def __init__(self, model: Model, device: str, threshold:float=0.5):
        self.model = model
        self.device = device
        self.threshold = threshold
        self.model.fuse()
        self.model.eval()

    def predict(self, image: Image.Image) -> list[DetectionObject]:
        results = self.model.predict(image, device=self.device, verbose=False)

        detections = [
            DetectionObject(
                        class_id=int(cls),
                        object_id=i,
                        label=self.model.names[int(cls)],
                        confidence=float(conf),
                        bbox=[float(x) for x in box],
            )
            for r in results
            for i, (box, cls, conf) in enumerate(zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf, strict=True))
        ]
        return list(filter(lambda det: det.confidence >= self.threshold, detections))

class RoboflowDetector(BaseDetector):
    def __init__(self, model: RFDETR, device: str, threshold:float=0.5):
        self.model = model
        self.dummy_device = device
        self.device = model.model.device # right now, i just let roboflow get the device
        # model.model.model.to(device)
        self.threshold = threshold
        self.model.optimize_for_inference()

    def predict(self, image: Image.Image) -> list[DetectionObject]:
        results = self.model.predict(image, threshold=self.threshold)
        detections = [
            DetectionObject(
                class_id=class_id,
                object_id=i,
                label=COCO_CLASSES.get(class_id),
                confidence=float(score),
                bbox=list(map(float, box))
            )
            for i, (box, score, class_id) in enumerate(zip(results.xyxy, results.confidence, results.class_id, strict=True))
        ]
        return detections

In [ ]:
from enum import StrEnum
from pathlib import Path
import urllib.request

from rfdetr import RFDETRMedium
from ultralytics import RTDETR
from ultralytics import YOLO


class DetectionModel(StrEnum):
    RT_DETR = "rt_detr"
    YOLO = "yolo"
    RF_DETR = "rf_detr"

class ModelManager:
    """Handles integrity, paths, and downloads for all object detection models."""
    MODELS_DIR = Path.cwd().parent.parent / "detection_models"

    # Registry of models used in the project; can be expanded
    REGISTRY = {
        DetectionModel.RT_DETR: "https://github.com/ultralytics/assets/releases/download/v8.3.0/rtdetr-x.pt",
        DetectionModel.YOLO: "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11x.pt",
        DetectionModel.RF_DETR: None,  # no weights file
    }

    @classmethod
    def get_model_path(cls, model_name: DetectionModel) -> Path:
        return cls.MODELS_DIR / cls.REGISTRY[model_name.value].split("/")[-1]

    @classmethod
    def ensure_model(cls, model_name: DetectionModel) -> Path | None:
        url = cls.REGISTRY[model_name]
        if url is None:
            return None

        path = cls.MODELS_DIR / Path(url).name
        if not path.exists():
            cls.MODELS_DIR.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, path)
        return path

    @classmethod
    def load_detector(
        cls,
        model_name: DetectionModel,
        device: str | None = None,
        threshold: float = 0.5
    ) -> BaseDetector:
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        path = cls.ensure_model(model_name)
        match model_name:
            case DetectionModel.YOLO:
                return UltralyticsDetector(YOLO(path), device, threshold)
            case DetectionModel.RT_DETR:
                return UltralyticsDetector(RTDETR(path), device, threshold)
            case DetectionModel.RF_DETR:
                return RoboflowDetector(RFDETRMedium(), device, threshold)

In [ ]:
import requests

image = Image.open(requests.get('https://i.guim.co.uk/img/media/7d04c4cb7510a4bd9a8bec449f53425aeccee895/298_266_1150_690/master/1150.jpg?width=1200&height=900&quality=85&auto=format&fit=crop&s=1e23629581e15bf38ee5a9f2a56e1489', stream=True).raw)
show(image)

In [ ]:
detector = ModelManager.load_detector(DetectionModel.YOLO, "cuda", 0.5)

In [ ]:
from PIL import Image
import torch


class DetectionService:
    """Stateless service for object detection using YOLO/RT-DETR."""
    # here it might depend on the backend whether ultralytics or roboflow... but that is easy to guess...
    def __init__(self, model_name: DetectionModel = DetectionModel.RF_DETR, model_path: Path | None = None, device: str | None = None, threshold: float = 0.5):
        self.model_path = ModelManager.ensure_model(model_name) if model_path is None else model_path
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Loading Detection Model: {self.model_path} on {self.device}")
        self.model = ModelManager.load_detector(model_name, self.device, threshold)

    def detect(self, image: Image.Image) -> list[DetectionObject]:
        return self.model.predict(image)


In [ ]:
ds = DetectionService(
    DetectionModel.RT_DETR,
    model_path=None,
    device="cuda",
    threshold=0.5
)

In [ ]:
detections = ds.detect(image)

## Scene Graph

### SoM Painter

In [ ]:
import cv2
import numpy as np
import supervision as sv


def bboxes_to_masks(image: Image.Image, bboxes: np.ndarray, scale: float = 0.25) -> np.ndarray:
    """
    Convert bounding boxes to approximate masks using GrabCut (scaled image for speed).
    Returns boolean masks of shape (n, H, W).
    """
    orig_img = np.array(image.convert("RGB"))
    H, W = orig_img.shape[:2]

    # Scale image
    img = cv2.resize(orig_img, (0, 0), fx=scale, fy=scale)
    Hs, Ws = img.shape[:2]

    masks = []

    for box in bboxes:
        x1, y1, x2, y2 = map(int, box)
        # Scale bbox to resized image
        rect = (int(x1 * scale), int(y1 * scale), int((x2 - x1) * scale), int((y2 - y1) * scale))

        mask = np.zeros((Hs, Ws), np.uint8)
        bgdModel = np.zeros((1, 65), np.float64)
        fgdModel = np.zeros((1, 65), np.float64)

        # GrabCut on scaled image
        cv2.grabCut(img, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)

        # Boolean mask on scaled image
        mask_bool = ((mask == 1) | (mask == 3))

        # Resize mask back to original size
        mask_orig_size = cv2.resize(mask_bool.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
        masks.append(mask_orig_size)

    return np.stack(masks, axis=0)


class SoMPainter:
    """Handles Set-of-Mark overlay on pictures for scene graph generation (https://som-gpt4v.github.io/)"""

    def __init__(self):
        self.box_annotator = sv.BoxAnnotator(
            thickness=2, color_lookup=sv.ColorLookup.INDEX
        )
        self.label_annotator = sv.LabelAnnotator(
            text_position=sv.Position.CENTER,
            text_color=sv.Color.BLACK,
            color=sv.Color.WHITE,
        )
        self.mask_annotator = sv.MaskAnnotator(
            color=sv.Color.BLACK,
            color_lookup=sv.ColorLookup.INDEX,
            opacity=0.5,
        )
        self.polygon_annotator = sv.PolygonAnnotator(
            color_lookup=sv.ColorLookup.INDEX,
            thickness=2,
        )

    def paint(
        self,
            image: np.ndarray | Image.Image,
            detections: list[DetectionObject],
            class_names: bool = False,
            bbox: bool = False,
            mask: bool = False,
            polygon: bool = False
    ) -> np.ndarray:
        """
        Applies SoM visualization to an image.
        """

        # start with 1 because 0 is never displayed somehow
        labels = [f"{(det.label + "_" if class_names else "") + str(det.object_id + 1)}" for det in detections]  # # was there
        xyxy = np.array([
            det.bbox for det in detections
        ])
        masks = bboxes_to_masks(image, xyxy) if mask or polygon else None
        conf = np.array([det.confidence for det in detections])
        ids = np.array([det.class_id for det in detections])
        detections = sv.Detections(
                xyxy=xyxy,
                confidence=conf,
                class_id=ids,
                mask=masks
        )
        annotated_image = (
            self.box_annotator.annotate(scene=image.copy(), detections=detections)
            if bbox
            else image.copy()
        )
        if mask:
            annotated_image = self.mask_annotator.annotate(scene=annotated_image, detections=detections)
        if polygon:
            annotated_image = self.polygon_annotator.annotate(scene=annotated_image, detections=detections)
        annotated_image = self.label_annotator.annotate(
            scene=annotated_image, detections=detections, labels=labels
        )
        return annotated_image


In [ ]:
painter = SoMPainter()

In [ ]:
#masks = bboxes_to_masks(image, [det.bbox for det in detections])

In [ ]:
som_image = painter.paint(image, detections, bbox=True, mask=False, polygon=True, class_names=True)
show(som_image)

### SGG

In [ ]:
from abc import ABC
from abc import abstractmethod
from enum import StrEnum


class VLMBackend(StrEnum):
    OPENAI = "openai"
    LOCAL = "local"
    LOCAL_4BIT = "local_4bit"

class BaseVLM(ABC):
    @abstractmethod
    async def infer(self, system_prompt: str, user_prompt: str, image: bytes) -> str:
        pass


In [ ]:
from typing import Any


class LLMLabelerConfig(BaseModel):
    """Configuration for the VLM Scene Graph ground truth Generator (ex. GPT-4o)."""
    backend: VLMBackend = VLMBackend.OPENAI
    # TODO: Right now, vendor-locked for OpenAI, might change for future
    model_id: str = "gpt-4o"
    path_to_model: Path | None = None
    temperature: float = Field(0.0, ge=0.0, le=2.0)
    max_tokens: int | None = Field(512, gt=0)

    system_prompt: str = (
        "You are a robotic scene graph generator. "
        "Analyze the provided Set-of-Mark (SoM) image where objects are marked with numerical IDs. "
        "Output a JSON list of spatial or other relationships "
        # "using ONLY the allowed predicates. " # Provide predicates into prompt if ClosedVocab, else dont mention
        "Unary predicates are represented by the same subject and object "
        "Format: [{'sub': 'ID', 'rel': 'PREDICATE', 'obj': 'ID'}]. "
        "Example: [{'sub': '1', 'rel': 'holding', 'obj': '2'}, {'sub': '1', 'rel': 'red', 'obj': '1'}]."
    )
    backend_kwargs: dict[str, Any] = Field(default_factory=dict)

In [ ]:
import base64

from openai import AsyncOpenAI


class OpenAIVLM(BaseVLM):
    def __init__(self, config: LLMLabelerConfig, openai_client_config: dict = None):
        if openai_client_config is None:
            openai_client_config = {}
        self.client = AsyncOpenAI(**openai_client_config)
        self.config = config

    async def infer(self, system_prompt:str, user_prompt:str, image: bytes) -> str:
        encoded = base64.b64encode(image).decode()
        response = await self.client.chat.completions.create(
            model=self.config.model_id,
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": user_prompt},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{encoded}"},
                        },
                    ],
                },
            ],
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
            response_format={"type": "json_object"},
        )
        return response.choices[0].message.content or ""

In [ ]:
import io

from PIL import Image
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import Qwen2VLForConditionalGeneration
from transformers import Qwen3VLForConditionalGeneration


class LocalHFVLM(BaseVLM):
    def __init__(self, model_id: str, device: str | None = None, dtype=None,
        attn_implementation: str = "flash_attention_2"):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        if "qwen2" in model_id.lower().strip():
            model_cls = Qwen2VLForConditionalGeneration
        elif "qwen3" in model_id.lower().strip():
            model_cls = Qwen3VLForConditionalGeneration
        else:
            model_cls = AutoModelForVision2Seq
        self.model = model_cls.from_pretrained(
            model_id,
            dtype=dtype or (
                torch.bfloat16 if device == "cuda" else torch.float32
            ),
            device_map="auto",
            attn_implementation=attn_implementation,
        )

        self.processor = AutoProcessor.from_pretrained(model_id)
        self.device = self.model.device

    async def infer(self, system_prompt: str, user_prompt: str, image: bytes) -> str:
        img = Image.open(io.BytesIO(image)).convert("RGB")

        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": img,
                    },
                    {"type": "text", "text": user_prompt},
                ],
            }
        ]
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        inputs = inputs.to(self.device)

        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=512)

        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out, strict=True)]

        return self.processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

class Local4BitVLM(BaseVLM):
    def __init__(self, model_id: str, trust_remote_code: bool = True, device_map: str = "auto"):
        self.model = AutoModelForVision2Seq.from_pretrained(
            model_id,
            device_map=device_map,
            trust_remote_code=trust_remote_code,
        )
        self.processor = AutoProcessor.from_pretrained(
            model_id,
            trust_remote_code=trust_remote_code,
        )

    async def infer(self, system_prompt: str, user_prompt: str, image: bytes) -> str:
        img = Image.open(io.BytesIO(image)).convert("RGB")

        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": user_prompt},
                ],
            },
        ]

        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=512)

        trimmed = [o[len(i):] for i, o in zip(inputs["input_ids"], out, strict=True)]

        return self.processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]


In [ ]:
from pydantic import BaseModel
from pydantic import Field
from pydantic import model_validator


class OntologyConfig(BaseModel):
    """Defines world model for teacher and student models in Object Detection FT process.
    Serves as ontology definition for predicates in VLM finetune too (optional param).
    """

    objects: dict[str, str] | None = Field(
        None, description="Map of more specific to more general"
    )
    predicates: list[str] | None = Field(
        None, description="List of predicates in VLM SGG finetuning."
    )

    @model_validator(mode="after")
    def check_at_least_one_ontology(self):
        if self.objects or self.predicates:
            return self
        raise ValueError(
            "At least one ontology is required (dict[str, str]). Specify objects or predicates (list[str])."
        )

In [ ]:
from dataclasses import dataclass
from dataclasses import field
import json
import logging
from pathlib import Path
import re


@dataclass
class SceneGraphEdge:
    sub: str
    rel: str
    obj: str

    def __hash__(self):
        return hash((self.sub, self.rel, self.obj))

    def __eq__(self, other):
        if not isinstance(other, SceneGraphEdge):
            return NotImplemented
        return (self.sub, self.rel, self.obj) == (other.sub, other.rel, other.obj)

@dataclass
class SceneGraph:
    """
    A structured scene graph object, returned by SceneGraphGenerator.generate.
    """
    edges: list[SceneGraphEdge] = field(default_factory=list)
    no_label_edges: list[SceneGraphEdge] = field(default_factory=list)
    raw: Any | None = None  # raw output from the VLM, useful for debugging

    def __post_init__(self):
        self.deduplicate()

    @staticmethod
    def _normalize_id(s: str) -> str:
        """Extract numeric ID from strings like 'cat_1' or return as is if already numeric"""
        match = re.search(r"(\d+)$", s)
        return match.group(1) if match else s

    @classmethod
    def from_list(cls, data: list[dict], raw: Any = None) -> "SceneGraph":
        edges = []
        for item in data:
            if all(k in item for k in ["sub", "rel", "obj"]):
                edges.append(SceneGraphEdge(sub=item["sub"], rel=item["rel"], obj=item["obj"]))
            else:
                # fallback for unexpected structure
                edges.append(SceneGraphEdge(
                    sub=str(item.get("sub", "")),
                    rel=str(item.get("rel", "")),
                    obj=str(item.get("obj", ""))
                ))
        no_label_edges = [
            SceneGraphEdge(
                sub=cls._normalize_id(edge.sub),
                rel=edge.rel,
                obj=cls._normalize_id(edge.obj),
            )
            for edge in edges
        ]
        return cls(edges=edges, no_label_edges=no_label_edges , raw=raw)

    def deduplicate(self):
        self.edges = list(set(self.edges))
        self.no_label_edges = list(set(self.no_label_edges))

    def subjects(self) -> list[str]:
        return [edge.sub for edge in self.edges]

    def objects(self) -> list[str]:
        return [edge.obj for edge in self.edges]

    def predicates(self) -> list[str]:
        return [edge.rel for edge in self.edges]

    def attributes(self) -> list[str]:
        return [edge.rel for edge in self.edges if edge.obj == edge.sub]

    def as_dict(self) -> list[dict]:
        return [{"sub": e.sub, "rel": e.rel, "obj": e.obj} for e in self.edges]

    def __len__(self):
        return len(self.edges)


class SceneGraphGenerator:
    def __init__(
        self, config: LLMLabelerConfig, ontology_config: OntologyConfig | None = None
    ):
        self.config = config
        self.ontology_config = ontology_config
        self.predicates = ontology_config.predicates if ontology_config else None
        self.vlm = self.build_vlm(config)

    @staticmethod
    def build_vlm(config: LLMLabelerConfig) -> BaseVLM:
        match config.backend:
            case VLMBackend.OPENAI:
                return OpenAIVLM(config, config.backend_kwargs)
            case VLMBackend.LOCAL:
                return LocalHFVLM(config.model_id, **config.backend_kwargs)
            case VLMBackend.LOCAL_4BIT:
                return Local4BitVLM(config.model_id, **config.backend_kwargs)

    @staticmethod
    def _encode_image(image_path: Path) -> str:
        with image_path.open("rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")

    @staticmethod
    def _to_bytes(image: Path | bytes | Image.Image | np.ndarray) -> bytes:
        """Convert input to bytes for VLM inference."""
        if isinstance(image, bytes):
            return image
        elif isinstance(image, Path):
            return image.read_bytes()
        elif isinstance(image, Image.Image):
            with io.BytesIO() as buf:
                image.save(buf, format="JPEG")
                return buf.getvalue()
        elif isinstance(image, np.ndarray):
            pil_img = Image.fromarray(image.astype("uint8"))
            with io.BytesIO() as buf:
                pil_img.save(buf, format="JPEG")
                return buf.getvalue()
        else:
            raise TypeError(
                f"Unsupported input type: {type(image)}. Must be Path, bytes, PIL.Image, or np.ndarray."
            )

    async def generate(self, image: Path | bytes | Image.Image | np.ndarray, verbose:bool=False) -> SceneGraph:
        image_bytes = self._to_bytes(image)

        system_prompt = self.config.system_prompt

        user_prompt = (
            "Allowed predicates: " + ", ".join(self.predicates)
            if self.predicates
            else "Focus on spatial, semantic, and functional relationships."
        )

        raw = await self.vlm.infer(system_prompt, user_prompt, image_bytes)
        if verbose:
            logger.info(f"VLM output: {raw}")
        try:
            data = json.loads(raw)

            if isinstance(data, dict):
                for key in ["relationships", "scene_graph", "triplets", "relations"]:
                    if key in data:
                        data = data[key]
                        break
            else:
                data = [data]
            if not isinstance(data, list):
                data = []
        except Exception:
            logger.warning("Failed to parse VLM output as JSON")
            data = []
        return SceneGraph.from_list(data, raw=raw)
    # async def batch_generate(self, image_paths: list[Path], batch_size: int = 100):
    #     semaphore = asyncio.Semaphore(batch_size)
    #
    #     async def limited_generate(path: Path):
    #         async with semaphore:
    #             result = await self.generate(path)
    #             return path, result
    #
    #     tasks = [limited_generate(p) for p in image_paths]
    #     logger.info(
    #         f"Starting batch generation for {len(tasks)} images with concurrency {batch_size}..."
    #     )
    #     results = await asyncio.gather(*tasks)
    #     return results


In [ ]:
cfg = LLMLabelerConfig(
    backend=VLMBackend.OPENAI,
    model_id="gpt-4o-mini",
    backend_kwargs={"api_key":"redacted"},
    temperature=0.0,
    max_tokens=1024
)
ontology_cfg = OntologyConfig(
    predicates=["carry", "wears", "behind", "in_front_of", "happy", "joyful", "plays_with", "on"]
)

In [ ]:
cfg2 = LLMLabelerConfig(
    backend=VLMBackend.LOCAL,
    model_id="Qwen/Qwen2-VL-2B-Instruct",
    temperature=0.0,
    max_tokens=1024,
)

In [ ]:
sgg = SceneGraphGenerator(cfg, ontology_cfg)

In [ ]:
graph = await sgg.generate(som_image, verbose=True)

In [ ]:
user_prompt = (
            "Allowed predicates: " + ", ".join(ontology_cfg.predicates)
            if sgg.predicates
            else "Focus on spatial, semantic, and functional relationships."
)
image_bytes = sgg._to_bytes(som_image)
raw = await sgg.vlm.infer(cfg2.system_prompt, user_prompt, image_bytes)

In [ ]:
import ast

data = ast.literal_eval(raw)
graph = SceneGraph.from_list(data)

In [ ]:
graph.edges

In [ ]:
show(som_image)

In [ ]:
import networkx as nx


def draw_scene_graph(sg: SceneGraph, figsize=(12, 6), node_color="lightblue"):
    """
    Draw a simple scene graph using networkx and matplotlib.

    Args:
        sg: SceneGraph object
        figsize: Figure size
        node_color: Node fill color
    """
    G = nx.DiGraph()

    # Add edges from the SceneGraph
    for edge in sg.edges:
        G.add_node(edge.sub)
        G.add_node(edge.obj)
        G.add_edge(edge.sub, edge.obj, label=edge.rel)

    pos = nx.spring_layout(G, seed=42)  # nice layout
    plt.figure(figsize=figsize)

    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color=node_color, node_size=1200)
    nx.draw_networkx_labels(G, pos, font_size=12)

    # Draw edges
    nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=35, edge_color="gray")
    edge_labels = nx.get_edge_attributes(G, "label")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=12)

    plt.axis("off")
    plt.show()

In [ ]:
draw_scene_graph(graph)

## ReID and Tracking

In [ ]:
import logging

import numpy as np
from numpy.typing import NDArray
from PIL import Image
from transformers import AutoModel
from transformers import CLIPImageProcessor


class FeatureExtractor:
    """Extracts visual embeddings (fingerprints) for ReID using Nvidia RADIO."""

    REPO = "nvidia/C-RADIOv4-SO400M"

    def __init__(self, device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if self.device == "cuda" else torch.float32

        logger.info(f"Loading Nvidia RADIO ({self.REPO}) on {self.device}...")
        self.model = (
            AutoModel.from_pretrained(
                self.REPO, trust_remote_code=True, torch_dtype=dtype
            )
            .to(self.device)
            .eval()
        )

        self.processor = CLIPImageProcessor.from_pretrained(
            self.REPO, trust_remote_code=True
        )

    def extract(self, image: Image.Image, detections: list[DetectionObject]) -> NDArray:
        """
        Crops the image based on detections and returns a batch of embeddings.
        Returns: (N, D) array where N=len(detections)
        """
        if not detections:
            return np.array([])

        # 1. Prepare Crops with FIXED Resolution
        # We must use a fixed size (e.g. 384x384) so we can batch them into one tensor.
        # 384 is a standard resolution that works well with C-RADIO.
        TARGET_SIZE = (384, 384)

        crops = []
        for det in detections:
            # bbox is [x1, y1, x2, y2]
            # Convert float bbox to int
            x1, y1, x2, y2 = map(int, det.bbox)

            # Clamp to image bounds to avoid errors
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(image.width, x2)
            y2 = min(image.height, y2)

            # Handle degenerate boxes (width or height 0)
            if x2 <= x1 or y2 <= y1:
                # Create a black dummy crop if detection is invalid
                crop = Image.new("RGB", TARGET_SIZE)
            else:
                crop = image.crop((x1, y1, x2, y2))
                crop = crop.resize(TARGET_SIZE, Image.Resampling.BICUBIC)

            crops.append(crop)

        # 2. Batch Inference
        with torch.no_grad():
            # We explicitly disable resizing in the processor since we did it manually
            inputs = self.processor(
                images=crops,
                return_tensors="pt",
                do_resize=False,
                do_center_crop=False
            )

            pixel_values = inputs.pixel_values.to(self.model.dtype).to(self.device)

            summary, _ = self.model(pixel_values)

            # 3. Normalize for Cosine Similarity
            summary = summary / summary.norm(p=2, dim=-1, keepdim=True)

        return summary.cpu().float().numpy()

In [ ]:
from dataclasses import dataclass
from dataclasses import field
import time

from scipy.optimize import linear_sum_assignment


@dataclass
class TrackedObject:
    """The persistent identity of a detected entity."""
    id: int
    label: str
    embedding: np.ndarray
    bbox: list[float]
    confidence: float

    # State
    last_seen: float = field(default_factory=time.time)
    first_seen: float = field(default_factory=time.time)
    hits: int = 1
    frames_since_seen: int = 0

    @property
    def center(self):
        """Returns (x_center, y_center)"""
        return ((self.bbox[0] + self.bbox[2])/2, (self.bbox[1] + self.bbox[3])/2)

    def update(self, det: DetectionObject, embedding: np.ndarray):
        """Update state with new observation."""
        self.bbox = det.bbox
        self.confidence = det.confidence

        # Smooth embedding (Exponential Moving Average) to stabilize identity
        # We give 10% weight to the new look, 90% to history
        self.embedding = 0.9 * self.embedding + 0.1 * embedding
        self.embedding /= np.linalg.norm(self.embedding)  # Re-normalize

        self.last_seen = time.time()
        self.hits += 1
        self.frames_since_seen = 0

class Associator:
    """Matches tracks with new detections."""
    def __init__(self, w_vis=0.8, w_geo=0.2, match_threshold=0.4):
        self.w_vis = w_vis
        self.w_geo = w_geo
        self.match_threshold = match_threshold
        self.INF = 1000.0

    def compute_cost(self, tracks: list[TrackedObject], detections: list[DetectionObject], embeddings: np.ndarray):
        """
        Creates a Cost Matrix where rows=tracks, cols=detections.
        Cost is low if they are the same object.
        """
        cost_matrix = np.zeros((len(tracks), len(detections)))

        for t_idx, track in enumerate(tracks):
            for d_idx, det in enumerate(detections):
                # 1. Hard Constraint: Labels must match (Context: "cat" cannot become "dog")
                if track.label != det.label:
                    cost_matrix[t_idx, d_idx] = self.INF
                    continue

                # 2. Visual Cost (Cosine Distance)
                # Dot product of normalized vectors = Cosine Similarity
                # Cost = 1 - Similarity
                det_emb = embeddings[d_idx]
                sim = np.dot(track.embedding, det_emb)
                vis_cost = 1.0 - sim

                # 3. Geometric Cost (Spatial Distance)
                # In Robot production code: Use Angle Difference
                # In Notebook: Use Center Distance Normalized by Image Size (approx 1000px)
                det_center = ((det.bbox[0]+det.bbox[2])/2, (det.bbox[1]+det.bbox[3])/2)
                dist = np.linalg.norm(np.array(track.center) - np.array(det_center))
                geo_cost = dist / 1000.0 # Normalize roughly

                # Weighted Sum
                cost_matrix[t_idx, d_idx] = (self.w_vis * vis_cost) + (self.w_geo * geo_cost)

        return cost_matrix

    def match(self, tracks: list[TrackedObject], detections: list[DetectionObject], embeddings: np.ndarray):
        if not tracks:
            return [], [], list(range(len(detections)))
        if not detections:
            return [], list(range(len(tracks))), []

        # Hungarian Algorithm
        cost_matrix = self.compute_cost(tracks, detections, embeddings)
        row_idx, col_idx = linear_sum_assignment(cost_matrix)

        matches = []
        matched_tracks = set()
        matched_dets = set()

        for r, c in zip(row_idx, col_idx, strict=False):
            if cost_matrix[r, c] < self.match_threshold:
                matches.append((r, c))
                matched_tracks.add(r)
                matched_dets.add(c)

        unmatched_tracks = [i for i in range(len(tracks)) if i not in matched_tracks]
        unmatched_dets = [i for i in range(len(detections)) if i not in matched_dets]

        return matches, unmatched_tracks, unmatched_dets

In [ ]:
class SceneMemory:
    """Manages the lifecycle of objects in the robot's world."""

    def __init__(self):
        self.tracks: dict[int, TrackedObject] = {}
        self.next_id = 1

        # Dependencies
        self.extractor = FeatureExtractor()
        self.associator = Associator()

    def update(self, image: Image.Image, detections: list[DetectionObject]):
        """
        Main pipeline step:
        1. Extract Embeddings
        2. Match to History
        3. Update IDs in DetectionObjects
        """
        if not detections:
            return detections

        # 1. Extract Embeddings
        embeddings = self.extractor.extract(image, detections)

        # 2. Match
        active_tracks_list = list(self.tracks.values())
        matches, un_tracks, un_dets = self.associator.match(active_tracks_list, detections, embeddings)

        # 3. Update Matched Tracks
        for t_idx, d_idx in matches:
            track = active_tracks_list[t_idx]
            det = detections[d_idx]
            emb = embeddings[d_idx]

            # Update Track State
            track.update(det, emb)

            # ASSIGN ID TO DETECTION (Critical for SoM!)
            det.object_id = track.id

        # 4. Create New Tracks
        for d_idx in un_dets:
            det = detections[d_idx]
            emb = embeddings[d_idx]

            new_track = TrackedObject(
                id=self.next_id,
                label=det.label,
                embedding=emb,
                bbox=det.bbox,
                confidence=det.confidence
            )
            self.tracks[self.next_id] = new_track

            # Assign ID
            det.object_id = self.next_id
            self.next_id += 1

        # 5. Prune (Simple logic for notebook)
        # In a real loop, you'd increment 'frames_since_seen' for un_tracks indices
        # and delete if > threshold.

        return detections

In [ ]:
memory = SceneMemory()

In [ ]:
detections = ds.detect(image)

In [ ]:
print(f"Raw Detections: {[d.label for d in detections]}")
print(f"IDs before memory: {[d.object_id for d in detections]}")

In [ ]:
detections_with_ids = memory.update(image, detections)

In [ ]:
detections_with_ids

In [ ]:
memory.tracks

# Pipeline

In [ ]:
from dataclasses import dataclass

import numpy as np
from PIL import Image


@dataclass
class PipelineResult:
    """Holds the complete state of a processed frame."""
    raw_image: Image.Image
    som_image: np.ndarray          # The image with tags drawn on it
    detections: list[DetectionObject] # List of objects with persistent IDs
    scene_graph: SceneGraph        # The semantic relationships

    def summary(self):
        print("--- Frame Summary ---")
        print(f"Objects Detected: {len(self.detections)}")
        print(f"Entities in Memory: {[d.object_id for d in self.detections]}")
        print(f"Relationships Found: {len(self.scene_graph.edges)}")
        for edge in self.scene_graph.edges:
            print(f"  - Object {edge.sub} {edge.rel} Object {edge.obj}")

In [ ]:
class VisualPipeline:
    def __init__(
        self,
        detector: DetectionService,
        memory: SceneMemory,
        painter: SoMPainter,
        sgg: SceneGraphGenerator
    ):
        self.detector = detector
        self.memory = memory
        self.painter = painter
        self.sgg = sgg

    async def process(self, image: Image.Image) -> PipelineResult:
        """
        Runs the full See-Track-Understand loop.
        """
        # 1. DETECT (Get raw boxes, no IDs yet)
        raw_detections = self.detector.detect(image)

        # 2. MEMORY (Assign Persistent IDs via ReID)
        # This modifies the detection objects in-place or returns new ones
        tracked_detections = self.memory.update(image, raw_detections)

        # 3. PAINT (Draw Set-of-Mark tags using the IDs)
        # We need numpy for the painter, but we keep PIL for the VLM if needed
        image_np = np.array(image)
        som_image = self.painter.paint(
            image_np,
            tracked_detections,
            bbox=True,
            mask=False,    # Turn on if you want segmentation masks
            polygon=False,
            class_names=True # Helps the VLM know "Object 1" is a "cup"
        )

        # 4. UNDERSTAND (Generate Scene Graph from SoM Image)
        # We pass the tagged image so the VLM can reference "Object 1"
        scene_graph = await self.sgg.generate(som_image, verbose=False)

        return PipelineResult(
            raw_image=image,
            som_image=som_image,
            detections=tracked_detections,
            scene_graph=scene_graph
        )

In [ ]:
memory = SceneMemory()
painter = SoMPainter()

# Create the pipeline
pipeline = VisualPipeline(ds, memory, painter, sgg) # sgg and reid can work in parallel, really

In [ ]:
result = await pipeline.process(image)

In [ ]:
result.summary()

In [ ]:
show(result.som_image, "Set-of-Mark Input")

In [ ]:
draw_scene_graph(result.scene_graph)

# RAG

In [ ]:

from pydantic import BaseModel
from pydantic import Field


class WorldEntity(BaseModel):
    """A simplified view of a tracked object for the LLM."""
    id: int
    label: str

    # We convert raw bbox/angle to human-readable text in the service
    # but we keep the raw values here if the LLM needs to do math.
    position_hint: str = Field(..., description="e.g. 'to your left', 'center', 'far right'")
    distance_estimate: str = Field(..., description="e.g. 'close', 'far'")

    # Aggregated knowledge
    attributes: list[str] = Field(default_factory=list) # ['red', 'tall']
    relations: list[str] = Field(default_factory=list)  # ['on table', 'holding cup']

class WorldState(BaseModel):
    """The Context Window Payload."""
    timestamp: float
    entities: list[WorldEntity]

    def to_context_string(self) -> str:
        """Converts state to a clean text block for the LLM."""
        if not self.entities:
            return "You see nothing around you right now."

        lines = ["Here is what you see in the world right now:"]
        for e in self.entities:
            # Basic Info
            desc = f"- Object #{e.id} is a {e.label}. It is {e.distance_estimate} {e.position_hint}."

            # Attributes
            if e.attributes:
                desc += f" It appears {', '.join(e.attributes)}."

            # Relations
            if e.relations:
                # e.g. "on table", "next to chair"
                rel_text = ", ".join(e.relations)
                desc += f" It is {rel_text}."

            lines.append(desc)

        return "\n".join(lines)

In [ ]:
from typing import Literal


class UnderstandingConfig(BaseModel):
    backend: Literal["openai", "local", "local_4bit"]
    model_id: str
    inference: dict = Field(default_factory=dict) # max_tokens, temp, etc
    ontology: OntologyConfig

In [ ]:
import logging
import os

logger = logging.getLogger(__name__)

class LLMClient:
    """
    A unified client for Text Generation (Chat).
    Supports OpenAI (GPT-4) and Local LLMs (via OpenAI-compatible API like vLLM/Ollama).
    """
    def __init__(self, config: UnderstandingConfig):
        self.config = config

        # Determine API Key and Base URL
        # If backend is 'local', we assume an OpenAI-compatible local server (e.g., localhost:8000)
        # If backend is 'openai', we use the real OpenAI API.

        api_key = os.getenv("OPENAI_API_KEY", "EMPTY")
        base_url = os.getenv("LLM_BASE_URL", None) # e.g., "http://localhost:11434/v1" for Ollama

        if config.backend == "openai":
            self.client = AsyncOpenAI(api_key=api_key)
        else:
            # For "local" or "local_4bit", we assume a local inference server
            # or we default back to OpenAI if no local URL is set.
            # You can customize this to load HuggingFace models directly if you prefer,
            # but keeping it API-based is cleaner for the server.
            self.client = AsyncOpenAI(api_key=api_key, base_url=base_url)

    async def generate_text(self, system_prompt: str, user_prompt: str) -> str:
        """
        Generates a text response based on the system and user prompts.
        """
        try:
            # Extract inference params with defaults
            max_tokens = self.config.inference.get("max_tokens", 512)
            temperature = self.config.inference.get("temperature", 0.7)

            response = await self.client.chat.completions.create(
                model=self.config.model_id,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )

            content = response.choices[0].message.content
            return content if content else "I'm not sure what to say."

        except Exception as e:
            logger.error(f"LLM Generation Error: {e}")
            return "I am having trouble connecting to my language center right now."

In [ ]:
import logging

logger = logging.getLogger(__name__)

class ChatService:
    def __init__(self, config: UnderstandingConfig, memory: SceneMemory):
        self.config = config
        self.memory = memory
        # Reuse your VLM client or create a dedicated text-only client
        self.llm = LLMClient(config)

    def _calculate_spatial_hint(self, angle_rad: float) -> str:
        """Converts radians to natural language."""
        # deg = math.degrees(angle_rad)
        # if -15 <= deg <= 15: return "directly in front of you"
        # if -45 <= deg < -15: return "slightly to your left"
        # if deg < -45:        return "far to your left"
        # if 15 < deg <= 45:   return "slightly to your right"
        # if deg > 45:         return "far to your right"
        return "nearby"

    def get_world_state(self) -> WorldState:
        """Snapshot the memory into a RAG-ready format."""
        entities = []

        # We access the internal tracks from memory
        # memory.tracks is Dict[int, TrackedObject]
        for track in self.memory.tracks.values():

            # Convert Internal Track -> LLM Entity
            entity = WorldEntity(
                id=track.id,
                label=track.label,
                position_hint=self._calculate_spatial_hint(getattr(track, 'angle', 0.0)),
                distance_estimate="nearby", # Placeholder until you have depth
                attributes=list(getattr(track, 'attributes', {}).keys()),
                relations=[f"{r['pred']} object #{r['obj_id']}" for r in getattr(track, 'relations', [])]
            )
            entities.append(entity)

        return WorldState(timestamp=0.0, entities=entities)

    async def chat(self, user_query: str) -> str:
        # 1. RAG: Retrieve Context
        state = self.get_world_state()
        context_str = state.to_context_string()

        # 2. Construct System Prompt
        system_prompt = (
            "You are Pepper, a helpful social robot assistant. "
            "You have a spatial memory of the world around you. "
            "Answer the user's question based strictly on the Context below. "
            "If the object is not in the context, say you don't know."
            "\n\n"
            f"--- CONTEXT ---\n{context_str}\n----------------"
        )

        # 3. Call LLM
        logger.info(f"Chat Context: {context_str}")
        response = await self.llm.generate_text(
            system_prompt=system_prompt,
            user_prompt=user_query
        )

        return response

In [ ]:
ucfg = UnderstandingConfig(
    backend="openai",
    model_id="gpt-4o-mini",
    ontology=ontology_cfg
)

In [ ]:
chat_service = ChatService(ucfg, memory)

In [ ]:
chat_service.get_world_state()

In [ ]:
await chat_service.chat("What do you see?")